In [385]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
import ast


In [386]:
ads_base = pd.read_csv('../data/ads.csv')
users_base = pd.read_csv('../data/users.csv')

users= users_base.copy()
ads = ads_base.copy()

In [387]:
users.head(1)

,user_id,age,gender,lat,lon,preferences
0,0,43,M,2.099503,-50.554383,"[('Game_Streaming', 0.024985108735607994), ('S..."


In [388]:
ads.head(1)

,ad_id,lat_target,lon_target,location_sensitive,age_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
0,0,-30.626167,-61.855339,1,"(18, 22)",Any,"{'Indie_Games': 0.06286300308346227, 'Game_Str...",16.366136,1636.613636,0.78569,0.030398,0.010648


In [389]:
ads["categories"].head()

0    {'Indie_Games': 0.06286300308346227, 'Game_Str...
1    {'PC_Gaming': 0.38503546915255144, 'MOBA': 0.6...
2    {'Game_Streaming': 0.5979574387468821, 'MOBA':...
3    {'Racing': 0.6596909459082921, 'Game_Streaming...
4                                  {'Simulation': 1.0}
Name: categories, dtype: object

In [390]:
type(users.preferences)

pandas.core.series.Series

In [391]:
# CSV perde a informação de tipo, então precisamos converter as strings de volta para listas e para dicionários. Para isso, podemos usar a função ast.literal_eval, que avalia uma string como uma expressão Python e retorna o objeto correspondente.

users["preferences"] = users["preferences"].apply(ast.literal_eval)
ads['categories'] = ads['categories'].apply(ast.literal_eval)

# Features Categorias

Gerar as categorias com seus pesos

OBS: isso foi feito porque para trabalhar com a abordagem de vetores, ADS e Users devem possuir mesma dimensões

Depois desenvolver uma forma mais inteligente para não fazer à mão

In [392]:
all_categories  = [
"FPS",
"Battle_Royale",
"MOBA",
"RPG",
"MMORPG",
"Strategy",
"Simulation",
"Sports_Games",
"Racing",
"Indie_Games",
"Retro_Gaming",
"Mobile_Gaming",
"PC_Gaming",
"Console_Gaming",
"Game_Development",
"Game_Streaming",
"Esports",
"Game_Hardware",
"VR_Gaming",
"Open_World",
"League_of_Legends",
"Fortnite",
"Minecraft",
"Counter_Strike",
"World_of_Warcraft"
]

In [393]:
# enumerar as categorias para criar um índice de categoria, que será usado para construir os vetores de características dos usuários e dos anúncios.
category_index = {cat: i for i, cat in enumerate(all_categories)}


In [394]:
category_index

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [395]:
# função para construir o vetor de características do usuário a partir de suas preferências, usando o índice de categoria para mapear as categorias para posições no vetor.

def build_user_vector(preferences, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in preferences:
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [396]:
# fazendo um teste para ver se a função build_user_vector está funcionando corretamente, construindo o vetor de características do primeiro usuário do dataset.
teste = build_user_vector(users.iloc[0]['preferences'], category_index)
display(teste)
display(users.iloc[0]['preferences'])
display(category_index)

array([0.        , 0.        , 0.01018336, 0.        , 0.        ,
       0.        , 0.39223273, 0.02069986, 0.        , 0.04843531,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.02498511, 0.00918893, 0.        , 0.02715214, 0.04884156,
       0.04954429, 0.        , 0.3687367 , 0.        , 0.        ])

[('Game_Streaming', 0.024985108735607994),
 ('Sports_Games', 0.020699862317418733),
 ('Minecraft', 0.36873670174426554),
 ('Indie_Games', 0.04843531447675644),
 ('MOBA', 0.010183361326199134),
 ('VR_Gaming', 0.02715213699237649),
 ('Esports', 0.009188926694486859),
 ('Open_World', 0.04884156481373497),
 ('Simulation', 0.3922327330940587),
 ('League_of_Legends', 0.049544289805095176)]

{'FPS': 0,
 'Battle_Royale': 1,
 'MOBA': 2,
 'RPG': 3,
 'MMORPG': 4,
 'Strategy': 5,
 'Simulation': 6,
 'Sports_Games': 7,
 'Racing': 8,
 'Indie_Games': 9,
 'Retro_Gaming': 10,
 'Mobile_Gaming': 11,
 'PC_Gaming': 12,
 'Console_Gaming': 13,
 'Game_Development': 14,
 'Game_Streaming': 15,
 'Esports': 16,
 'Game_Hardware': 17,
 'VR_Gaming': 18,
 'Open_World': 19,
 'League_of_Legends': 20,
 'Fortnite': 21,
 'Minecraft': 22,
 'Counter_Strike': 23,
 'World_of_Warcraft': 24}

In [397]:
len(teste)

25

In [398]:
# função análoga para o ads
# feito isso pq o ads foi salvo como dict e user como list

def build_ad_vector(categories, category_index):

    vector = np.zeros(len(category_index))

    for category, weight in categories.items():
        if category in category_index:
            vector[category_index[category]] = weight

    return vector

In [399]:
# teste para ads
teste_ad = build_ad_vector(ads.iloc[0]['categories'], category_index)
display(teste_ad)
display(ads.iloc[0]['categories'])


array([0.        , 0.        , 0.        , 0.8260921 , 0.        ,
       0.        , 0.        , 0.02873019, 0.        , 0.062863  ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.08231471, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

{'Indie_Games': 0.06286300308346227,
 'Game_Streaming': 0.08231470720072336,
 'RPG': 0.8260920989195466,
 'Sports_Games': 0.02873019079626784}

In [400]:
# aplcar as funções para construir os vetores de características para todos os usuários e anúncios, e armazenar esses vetores em novas colunas nos dataframes.

users['feature_vector'] = users['preferences'].apply(lambda prefs: build_user_vector(prefs, category_index))
ads['feature_vector'] = ads['categories'].apply(lambda cats: build_ad_vector(cats, category_index))

In [401]:
users.head(1)

,user_id,age,gender,lat,lon,preferences,feature_vector
0,0,43,M,2.099503,-50.554383,"[(Game_Streaming, 0.024985108735607994), (Spor...","[0.0, 0.0, 0.010183361326199134, 0.0, 0.0, 0.0..."


# Gerar os vetores finais 

- vai fazer a similaridade tudo junto ou vai separar??

In [402]:
# definir ordem fixa das features

CATEGORY_COLUMNS = all_categories

GENDER_COLUMNS = ["gender_F", "gender_M"]

In [403]:
# transformar features_vector em colunas separadas
category_features_users = pd.DataFrame(
    users["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

category_features_ads = pd.DataFrame(
    ads["feature_vector"].tolist(),
    columns=CATEGORY_COLUMNS
)

users = pd.concat([users, category_features_users], axis=1)
ads = pd.concat([ads, category_features_ads], axis=1)

In [404]:
# garantir que todas as colunas de cidade, estado e gênero estejam presentes em ambos os dataframes, preenchendo com zeros onde necessário
for col in GENDER_COLUMNS:
    if col not in users:
        users[col] = 0
    if col not in ads:
        ads[col] = 0

# Gerar os vetores finais sem cidade/estado

In [405]:
category_features_users

,FPS,Battle_Royale,MOBA,RPG,MMORPG,Strategy,Simulation,Sports_Games,Racing,Indie_Games,...,Game_Streaming,Esports,Game_Hardware,VR_Gaming,Open_World,League_of_Legends,Fortnite,Minecraft,Counter_Strike,World_of_Warcraft
0,0.000000,0.0,0.010183,0.000000,0.000000,0.000000,0.392233,0.0207,0.0,0.048435,...,0.024985,0.009189,0.000000,0.027152,0.048842,0.049544,0.000000,0.368737,0.000000,0.000000
1,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008369,0.000000
2,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.182170,0.000000,0.183739,0.000000,0.184107,0.124182,0.150162
3,0.000000,0.0,0.000000,0.019819,0.000000,0.485650,0.150442,0.0000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.082305,0.000000,0.000000,0.261784
4,0.000000,0.0,0.000000,0.627116,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,...,0.000000,0.000000,0.143685,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,...,0.347001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9996,0.000000,0.0,0.000000,0.000000,0.134468,0.122400,0.088767,0.0000,0.0,0.000000,...,0.000000,0.003148,0.262351,0.000000,0.000000,0.000000,0.333126,0.000000,0.049139,0.000000
9997,0.000000,0.0,0.000000,0.000000,0.000000,0.106062,0.000000,0.0000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.892759,0.000000,0.000000,0.000000,0.000000,0.000000
9998,0.026058,0.0,0.000000,0.246518,0.000000,0.377660,0.000000,0.0000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.122328,0.000000,0.000000,0.170901


In [406]:
users.iloc[0].preferences

[('Game_Streaming', 0.024985108735607994),
 ('Sports_Games', 0.020699862317418733),
 ('Minecraft', 0.36873670174426554),
 ('Indie_Games', 0.04843531447675644),
 ('MOBA', 0.010183361326199134),
 ('VR_Gaming', 0.02715213699237649),
 ('Esports', 0.009188926694486859),
 ('Open_World', 0.04884156481373497),
 ('Simulation', 0.3922327330940587),
 ('League_of_Legends', 0.049544289805095176)]

In [407]:
users[category_features_users.columns].iloc[0]

FPS                  0.000000
Battle_Royale        0.000000
MOBA                 0.010183
RPG                  0.000000
MMORPG               0.000000
Strategy             0.000000
Simulation           0.392233
Sports_Games         0.020700
Racing               0.000000
Indie_Games          0.048435
Retro_Gaming         0.000000
Mobile_Gaming        0.000000
PC_Gaming            0.000000
Console_Gaming       0.000000
Game_Development     0.000000
Game_Streaming       0.024985
Esports              0.009189
Game_Hardware        0.000000
VR_Gaming            0.027152
Open_World           0.048842
League_of_Legends    0.049544
Fortnite             0.000000
Minecraft            0.368737
Counter_Strike       0.000000
World_of_Warcraft    0.000000
Name: 0, dtype: float64

In [408]:
FINAL_FEATURE_COLUMNS = (
    CATEGORY_COLUMNS
)

In [409]:
user_matrix = users[FINAL_FEATURE_COLUMNS].values
ad_matrix = ads[FINAL_FEATURE_COLUMNS].values

# PRECISA NORMALIZAR???

In [410]:
user_matrix

array([[0.        , 0.        , 0.01018336, ..., 0.3687367 , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00836938,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.18410717, 0.12418159,
        0.15016166],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.02605786, 0.        , 0.        , ..., 0.        , 0.        ,
        0.17090077],
       [0.03742659, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [411]:
similarity_matrix = cosine_similarity(user_matrix, ad_matrix)

In [412]:
similarity_matrix
# cada linha representa um usuário e cada coluna representa um anúncio, e os valores são as similaridades entre eles.

array([[1.25051171e-02, 1.57852015e-02, 4.83115667e-02, ...,
        2.64015183e-02, 0.00000000e+00, 4.30153434e-02],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 7.01037232e-03, 0.00000000e+00, ...,
        1.52265559e-01, 0.00000000e+00, 2.67424646e-01],
       ...,
       [0.00000000e+00, 6.95828474e-04, 0.00000000e+00, ...,
        5.34671818e-04, 0.00000000e+00, 2.46562790e-05],
       [4.87509189e-01, 5.98307748e-02, 0.00000000e+00, ...,
        3.18128614e-01, 5.19657938e-02, 1.53660501e-01],
       [9.59636497e-01, 0.00000000e+00, 0.00000000e+00, ...,
        2.66899815e-01, 4.83116545e-02, 2.34674981e-02]])

## Validar os resultados

In [413]:
user_index = 0

user_lat = users.iloc[user_index]["lat"]
user_lon = users.iloc[user_index]["lon"]

ads["distance"] = np.sqrt(
    (ads["lat_target"] - user_lat)**2 +
    (ads["lon_target"] - user_lon)**2
)

ads["distance_score"] = 1 / (1 + ads["distance"])

ads.loc[
    ads["location_sensitive"] == 0,
    "distance_score"
] = 1

In [414]:
import ast

ads["age_target"] = ads["age_target"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

user_gender = users.iloc[user_index]["gender"]
user_age = users.iloc[user_index]["age"]

scores = similarity_matrix[user_index]

distance_scores = ads["distance_score"].values

final_scores = scores * distance_scores

ads_filtered = ads[
    (ads["gender_target"].isin([user_gender, "Any"])) &
    (
        ads["age_target"].apply(
            lambda age_range: True if age_range == 0 else age_range[0] <= user_age <= age_range[1]
        )
    )
]

filtered_scores = final_scores[ads_filtered.index]

top_ads = ads_filtered.index[filtered_scores.argsort()[::-1][:100]]

In [415]:
top_ads

Index([439, 660, 690, 309, 872, 288, 966, 122, 617, 542, 363, 548,  12, 186,
       508, 265, 225,  93, 117, 372, 171, 659, 445, 121,  91,  64, 232, 946,
       423, 831,  47, 346, 777, 192, 327, 688, 235, 768, 474, 364, 131, 957,
        15, 103, 326, 373, 793, 866, 377, 802,  52, 654, 884,  98, 233, 291,
       389, 658, 283, 999, 534, 136, 792, 740, 944, 310, 330, 860, 353, 412,
       254,  83, 972, 769, 269, 525, 657, 110, 349, 627, 224, 471, 378, 855,
       397, 963, 403, 670, 447, 200, 813, 762, 500, 492, 379, 375, 710, 532,
       565, 505],
      dtype='int64')

In [416]:
scores[top_ads]

array([0.71733922, 0.716586  , 0.71059196, 0.66171332, 0.65855431,
       0.65460971, 0.53947864, 0.45425541, 0.45360062, 0.4232558 ,
       0.4139771 , 0.38941316, 0.36913109, 0.34216276, 0.34205803,
       0.33169393, 0.29845766, 0.28695584, 0.27502277, 0.18424112,
       0.13537558, 0.12786758, 0.12724917, 0.62489184, 0.11334961,
       0.10196246, 0.09644896, 0.67436824, 0.09060963, 0.09041679,
       0.0899507 , 0.08853605, 0.08850685, 0.71733922, 0.08312254,
       0.0828269 , 0.08270902, 0.62012233, 0.08163361, 0.64402685,
       0.07931302, 0.0788971 , 0.07546957, 0.07317528, 0.06848941,
       0.06692503, 0.06645001, 0.06280677, 0.06190819, 0.06141066,
       0.06060918, 0.0582735 , 0.60568047, 0.71733922, 0.05307256,
       0.05173985, 0.04965749, 0.04561098, 0.04500812, 0.04301534,
       0.69725033, 0.04053323, 0.671359  , 0.03825989, 0.64821382,
       0.03648351, 0.0346755 , 0.03393416, 0.03224565, 0.2982199 ,
       0.02858197, 0.02730079, 0.629084  , 0.4932878 , 0.36080

In [417]:
ads_base.iloc[top_ads]

,ad_id,lat_target,lon_target,location_sensitive,age_target,gender_target,categories,bid,budget,trust_score,estimated_conversion_rate,click_probability
439,439,-14.615197,-61.287691,0,0,M,{'Simulation': 0.9999999999999999},15.000000,15000.000000,0.924901,0.020111,0.018720
660,660,1.311202,-66.661347,0,"(43, 47)",M,"{'Fortnite': 0.05268018162483455, 'PC_Gaming':...",24.303578,12151.788908,0.428560,0.013520,0.005124
690,690,-19.606453,-40.101304,0,0,Any,"{'Counter_Strike': 0.12136802699272922, 'Simul...",15.000000,1500.000000,0.446272,0.020835,0.014892
309,309,-27.744753,-40.722836,0,0,Any,"{'Open_World': 0.01871759482680016, 'Simulatio...",20.610971,20610.971355,0.913032,0.034520,0.021092
872,872,-18.854482,-44.443927,0,0,M,"{'Game_Hardware': 0.05808043799168352, 'Indie_...",15.000000,1500.000000,0.984166,0.022838,0.037847
...,...,...,...,...,...,...,...,...,...,...,...,...
375,375,-30.319899,-64.062611,1,0,Any,"{'Minecraft': 0.6739801137509406, 'Strategy': ...",15.000000,7500.000000,0.812826,0.022764,0.041171
710,710,3.318497,-55.931512,0,"(43, 47)",Any,{'Esports': 1.0},40.293898,40293.897959,0.718696,0.021663,0.008927
532,532,2.552859,-46.948520,0,0,M,{'Esports': 1.0},24.820091,2482.009066,0.618945,0.012104,0.032022
565,565,-20.454762,-67.598775,0,0,M,{'Esports': 1.0},18.793045,18793.045272,0.777406,0.029691,0.039511


In [418]:
users.loc[0]

user_id                                                              0
age                                                                 43
gender                                                               M
lat                                                           2.099503
lon                                                         -50.554383
preferences          [(Game_Streaming, 0.024985108735607994), (Spor...
feature_vector       [0.0, 0.0, 0.010183361326199134, 0.0, 0.0, 0.0...
FPS                                                                0.0
Battle_Royale                                                      0.0
MOBA                                                          0.010183
RPG                                                                0.0
MMORPG                                                             0.0
Strategy                                                           0.0
Simulation                                                    0.392233
Sports

# **OBJETIVO : score final**

score =
interest_similarity
× demographic_score
× bid
× trust_score
× estimated_conversion_rate

NÃO ESQUECER: incluir as